In [49]:
import os
import json
import scipy.interpolate
import numpy as np
import torch
import torchaudio.functional as F
import matplotlib.pyplot as plt
import time
import whisper

import json
import re
from IPython.display import HTML, display
import asyncio
from fuxi_api import FuxiAPI

In [50]:
def format_transcript_for_llm(result):
    """
    Convert Whisper output into a clean, timestamped transcript string 
    that an LLM can easily read and reference.
    """
    lines = []
    for seg in result["segments"]:
        start = seg["start"]
        end = seg["end"]
        text = seg["text"].strip()
        
        # Format as [MM:SS - MM:SS] text
        start_fmt = f"{int(start // 60):02d}:{int(start % 60):02d}"
        end_fmt = f"{int(end // 60):02d}:{int(end % 60):02d}"
        lines.append(f"[{start_fmt} - {end_fmt}] {text}")
    
    transcript = "\n".join(lines)
    return transcript

# Helper function to convert "MM:SS" to integer seconds
def time_to_seconds(time_str):
    # Strip out any accidental brackets the LLM might include (e.g., "[00:15]")
    clean_time = time_str.replace('[', '').replace(']', '').strip()
    try:
        m, s = clean_time.split(':')
        return int(m) * 60 + int(s)
    except ValueError:
        print(f"⚠️ Warning: Could not parse timestamp '{time_str}'. Defaulting to 0.")
        return 0

In [51]:
t1 = time.time()
model = whisper.load_model("large-v3-turbo")
file_path = "../downloads/trump_vids/slGLHD5VE00.mp4"
result = model.transcribe(file_path, language="en")
t2 = time.time()
print(t2 - t1)
print(result['segments'][0]['text'])

322.1548614501953
 Members of Congress, I have the high privilege and distinct honor of presenting to you the


In [55]:
transcript = format_transcript_for_llm(result)

In [56]:
api = FuxiAPI()  # Uses default config

print(f"🔧 Model:    {api.model_name}")
print(f"🔧 Endpoint: {api.end_point}")
print("-" * 50)

### CONSERVATIVE ###
# 1. Select your mode here: 'conservative' or 'liberal'
analysis_mode = "liberal" 

# 2. Define the specialized personas and tasks
POLITICAL_PROMPTS = {
    "conservative": (
        "You are an expert political strategist specializing in conservative messaging.\n"
        "Task: Highlight the following segments of this speech to align strongly with Republican and conservative values, "
        "supporting Donald Trump's policy agenda (e.g., economic nationalism, border security). \n"
        "Then, highlight the three most impactful segments from your rewrite and provide strategic reasons "
        "why they will resonate with a conservative voter base."
    ),
    "liberal": (
        "You are an expert political strategist specializing in progressive messaging.\n"
        "Task: Highlight the following segments of this speech to align strongly with Democratic and liberal values, "
        "critiquing or opposing Donald Trump's policy agenda (e.g., climate change, wealth equality). \n"
        "Then, highlight the three most impactful segments from your rewrite and provide strategic reasons "
        "why they will resonate with a progressive voter base."
    )
}

json_format = (
    "\n\nSTYLE GUIDELINE FOR TITLE:\n"
    "The 'title' field MUST be a highly engaging, TikTok-style 'hook' caption. "
    "Be creative and highly strategic based on your assigned persona. "
    "Use punctuation (like quotation marks or bolding) to reframe statements (e.g., to emphasize a point or imply skepticism/sarcasm). "
    "You may include emojis to actively show strong support, outrage, or disbelief, perfectly matching your political alignment.\n\n"
    "You MUST STRICTLY format your entire response strictly as a valid JSON object. "
    "Do not include any markdown, preamble, or conversational text. "
    "Use this exact schema:\n"
    "{\n"
    '  "summary": "A concise summary of the overall speech",\n'
    '  "key_topics": ["Topic 1", "Topic 2"],\n'
    '  "highlights": [\n'
    "    {\n"
    '      "title": "Your TikTok-style hook with emojis and framing punctuation",\n'
    '      "start_timestamp": "00:00", // Exact MM:SS string from the transcript\n'
    '      "end_timestamp": "00:33",   // Exact MM:SS string from the transcript\n'
    '      "rationale": "Your strategic reason for choosing this segment"\n'
    "    }\n"
    "  ]\n"
    "}\n\n"
)

# 3. Build the final instruction block dynamically (overwriting any previous memory)
if analysis_mode in POLITICAL_PROMPTS:
    # Apply the selected political bias and enforce an output format
    base_instruction = POLITICAL_PROMPTS[analysis_mode]
    format_requirements = (
        "\n\nPlease format your response exactly as follows:\n"
        "1. **Top 3 Highlighted Segments & Rationale**\n"
        "2. **Key Topics**\n"
        "3. **Summary**\n\n"
    )
    INSTRUCTION = base_instruction + json_format
else:
    # Fallback to the Neutral/Bipartisan default
    INSTRUCTION = (
        "You are a NEUTRAL, BIPARTISAN political speech analyst. "
        "Please analyze the speech and provide:\n"
        "1. **Key Topics** — Main themes discussed, with relevant timestamps\n"
        "2. **Notable Quotes** — Important or viral-worthy statements\n"
        "3. **Highlights** — The most engaging/important moments with timestamps\n"
        "4. **Summary** — A concise summary of the entire speech\n\n"
    )
    
# 4. Inject the transcript
FINAL_PAYLOAD = f"{INSTRUCTION}--- TRANSCRIPT ---\n{transcript}\n--- END TRANSCRIPT ---"

# --- Test 1: Execute the API Call ---
print(f"\n📝 Test 1: get_response ({analysis_mode} mode)")

# Assuming 'output' was a preamble string from your original code, prepend it here if needed
response = await api.get_response(FINAL_PAYLOAD) 
print(f"Response:\n{response}")

api.close()

🔧 Model:    gpt-4.1
🔧 Endpoint: http://aigc-api.apps-hangyan.danlu.netease.com/api/v2/text/chat
--------------------------------------------------

📝 Test 1: get_response (liberal mode)
✅ Tokens — prompt: 27281, completion: 525, total: 27806
Response:
{
  "summary": "Trump's speech touts a 'golden age' of America, celebrating economic growth, military victories, and hardline border policies, while forcefully attacking Democratic priorities. He claims record drops in crime, attacks progressive policies like DEI, and proposes privatized solutions for education and healthcare, all while scapegoating immigrants and dismissing climate action.",
  "key_topics": ["Economic Policy & Wealth Inequality", "Climate Change & Environmental Policy"],
  "highlights": [
    {
      "title": "“Did Someone Say ‘Golden Age’—But For Billionaires?! 👀💸”",
      "start_timestamp": "01:28",
      "end_timestamp": "02:37",
      "rationale": "This segment is where Trump frames his policies as creating a 'golden

<coroutine object FuxiAPI.close at 0x777fda71b340>

In [58]:
# 1. Clean the LLM output (removes markdown code blocks if present)
clean_response = re.sub(r'^```(?:json)?\n|\n```$', '', response.strip(), flags=re.MULTILINE)

try:
    # 2. Parse the JSON text into a Python dictionary
    analysis_data = json.loads(clean_response)
    
    print(f"📌 Summary: {analysis_data.get('summary')}")
    print("-" * 50)
    
    # 3. Loop through the highlights and generate a video player for each
    for i, highlight in enumerate(analysis_data.get("highlights", [])):
        # Extract the MM:SS strings and convert them using our helper function
        raw_start = highlight["start_timestamp"]
        raw_end = highlight["end_timestamp"]
        
        tstart = time_to_seconds(raw_start) - 1
        tend = time_to_seconds(raw_end) + 1
        
        title = highlight["title"]
        rationale = highlight["rationale"]
        
        # Build the HTML template dynamically
        html_code = f'''
        <div style="margin-bottom: 30px; border: 1px solid #ddd; padding: 15px; border-radius: 8px;">
            <h3>🎥 Clip {i+1}: {title}</h3>
            <p><strong>Strategic Rationale:</strong> {rationale}</p>
            <video width="720" controls>
              <source src="{file_path}#t={tstart},{tend}" type="video/mp4">
            </video>
            <p style="color: gray; font-size: 0.9em;">
              Original Timestamp: {raw_start} → {raw_end} (Playing: {tstart}s → {tend}s) (Clip time: {tend - tstart}s)
            </p>
        </div>
        '''
        
        # Display each HTML block
        display(HTML(html_code))

except json.JSONDecodeError as e:
    print("❌ Error: Could not parse the LLM response as JSON. Here is the raw output:")
    print(response)
except KeyError as e:
    print(f"❌ Error: The JSON is missing a required key: {e}")

📌 Summary: Trump's speech touts a 'golden age' of America, celebrating economic growth, military victories, and hardline border policies, while forcefully attacking Democratic priorities. He claims record drops in crime, attacks progressive policies like DEI, and proposes privatized solutions for education and healthcare, all while scapegoating immigrants and dismissing climate action.
--------------------------------------------------
